# Week 3 — Data preprocessing

This notebook prepares California single-family CRMLS closed sales for a price-prediction baseline. It contains its cleaning, feature policy, and model code, so it runs in a fresh checkout with the raw sales files and Python packages. It compares the number of immediately preceding training months using **pre-test rolling validation**, reserves the **most recent available month** for one final test, and writes `csv/cleaned_sales.csv`.

Run from the repository root with pandas, NumPy, and scikit-learn 1.3 or newer. The raw `csv/CRMLSSold*.csv` files are local inputs. `ClosePrice` is the label; `CloseDate` is split metadata and supplies as-of calendar features. Neither is passed to the estimator. This Week 3 baseline uses property and as-of features; the separate production evaluation also uses point-in-time comparable-sale features.

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 1))
ROOT = Path.cwd()
if not (ROOT / 'csv').is_dir():
    raise RuntimeError('Run this notebook from the repository root with csv/CRMLSSold*.csv inputs.')

pd.set_option('display.max_rows', 40)
print(f'Repository: {ROOT}')

Repository: C:\Users\Yingjie Huang\Downloads\idx-exchange


### Self-contained preprocessing code

The definitions below are embedded from the project's feature-policy and valuation utilities. They are included in this notebook so the deliverable does not depend on separate local Python modules.

In [2]:
"""Central policy for features used by the off-market valuation model.

Every model input must be available for an arbitrary California single-family
home at prediction time. The allowlist below is intentionally closed: adding a
new input requires an explicit policy review instead of silently training on
every column present in an MLS export.

``ClosePrice`` is the supervised-learning target and must never be an input.
``CloseDate`` is metadata; for historical rows it may supply the as-of context
used to build ``AsOf*`` features, but it must not be passed to the model.

The ``Comp*`` features are compliant only when they are computed from other
properties' sales known strictly before the subject row's as-of date. The
subject sale and all post-as-of transactions must be excluded.
"""

from __future__ import annotations

import re
from collections.abc import Iterable


TARGET_COLUMN = "ClosePrice"
METADATA_COLUMNS = frozenset({"CloseDate"})

# This is the complete production feature contract. Do not replace it with an
# exclusion list: new source columns must fail closed until they are reviewed.
MODEL_FEATURE_COLUMNS: tuple[str, ...] = (
    "LivingArea",
    "AboveGradeFinishedArea",
    "BelowGradeFinishedArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "PropertyAge",
    "Latitude",
    "Longitude",
    "AssociationFee",
    "GarageSpaces",
    "ParkingTotal",
    "FireplacesTotal",
    "Stories",
    "MainLevelBedrooms",
    "ViewYN",
    "WaterfrontYN",
    "BasementYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
    "CountyOrParish",
    "City",
    "PostalCode",
    "ElementarySchoolDistrict",
    "MiddleOrJuniorSchoolDistrict",
    "HighSchoolDistrict",
    "Levels",
    "AsOfYear",
    "AsOfMonthSin",
    "AsOfMonthCos",
    "CompMedianClosePrice",
    "CompMedianClosePricePerSqFt",
    "CompSaleCount",
    "CompDaysSinceLatestSale",
    "GeoCompMedianClosePrice",
    "GeoCompMedianClosePricePerSqFt",
    "GeoCompSaleCount",
    "GeoCompDaysSinceLatestSale",
    "ShiftGeoCompMedianClosePrice",
    "ShiftGeoCompMedianClosePricePerSqFt",
    "ShiftGeoCompSaleCount",
    "ShiftGeoCompDaysSinceLatestSale",
)

_MODEL_FEATURE_COLUMN_SET = frozenset(MODEL_FEATURE_COLUMNS)

# These two fields contain close-price terminology, but they are explicitly
# permitted because they are point-in-time aggregates of eligible prior comps.
_SAFE_COMP_PRICE_COLUMNS = frozenset(
    {
        "CompMedianClosePrice",
        "CompMedianClosePricePerSqFt",
        "GeoCompMedianClosePrice",
        "GeoCompMedianClosePricePerSqFt",
        "ShiftGeoCompMedianClosePrice",
        "ShiftGeoCompMedianClosePricePerSqFt",
    }
)

FORBIDDEN_EXACT_COLUMNS = frozenset(
    {
        "ListPrice",
        "OriginalListPrice",
        "DaysOnMarket",
        "CumulativeDaysOnMarket",
        "PurchaseContractDate",
        "ListingContractDate",
        "ClosePrice",
        "CloseDate",
    }
)

_NON_ALPHANUMERIC = re.compile(r"[^a-z0-9]+")
_CAMEL_CASE_BOUNDARY = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")

_LIST_PRICE_MARKERS = (
    "listprice",
    "listingprice",
    "originallistprice",
    "initiallistprice",
    "askingprice",
    "askprice",
)

_LIST_PRICE_EXACT_ALIASES = frozenset(
    {"lp", "olp", "listamount", "askingamount", "originalprice"}
)

_DOM_MARKERS = (
    "daysonmarket",
    "cumulativedaysonmarket",
    "cumulativemarketdays",
    "totaldaysonmarket",
    "marketingdays",
)

_CONTRACT_DATE_MARKERS = (
    "purchasecontractdate",
    "purchaseagreementdate",
    "listingcontractdate",
    "listcontractdate",
    "listingdate",
    "listeddate",
    "listdate",
    "undercontractdate",
    "pendingdate",
    "offeracceptancedate",
    "acceptedofferdate",
    "contractdate",
    "contractsigneddate",
    "contractstatuschangedate",
    "listingstartdate",
    "onmarketdate",
    "originalonmarketdate",
    "pendingtimestamp",
    "contingentdate",
)

_CLOSE_DATE_MARKERS = (
    "closedate",
    "closingdate",
    "saledate",
    "solddate",
    "settlementdate",
    "closeyear",
    "closequarter",
    "closemonth",
    "closeday",
    "closeweekday",
    "saleyear",
    "salequarter",
    "salemonth",
    "saleday",
    "saleweekday",
    "soldyear",
    "soldquarter",
    "soldmonth",
    "closeofescrowdate",
    "coedate",
    "recordingdate",
    "transferdate",
)

_TARGET_PRICE_MARKERS = (
    "closeprice",
    "saleprice",
    "soldprice",
    "finalsaleprice",
    "transactionprice",
    "settlementprice",
    "closingprice",
    "salesprice",
    "finalprice",
    "saleamount",
    "soldamount",
    "closeamount",
    "transactionamount",
)

_PPSF_MARKERS = (
    "ppsf",
    "pricepersquarefoot",
    "pricepersqft",
    "pricepersqfoot",
    "pricepersf",
)

_CLOSE_TO_LIST_MARKERS = (
    "closetolist",
    "saletolist",
    "soldtolist",
    "closetoasking",
    "saletoasking",
    "soldtoasking",
    "discountfromlist",
    "discountfromasking",
    "overasking",
    "splpratio",
    "sptolp",
    "sp2lp",
    "saletoask",
    "soldtoask",
)


class FeaturePolicyError(ValueError):
    """Raised when requested model inputs violate the leakage policy."""


def find_forbidden_features(columns: Iterable[str]) -> dict[str, str]:
    """Return detected leakage-prone columns and their policy reasons."""
    if isinstance(columns, (str, bytes)):
        raise TypeError("columns must be an iterable of column-name strings, not a string")

    violations: dict[str, str] = {}
    for column in columns:
        if not isinstance(column, str):
            raise TypeError("every feature name must be a string")
        reason = _forbidden_reason(column)
        if reason is not None:
            violations[column] = reason
    return violations


def _normalize_column_name(column: str) -> str:
    """Return a case- and separator-insensitive representation of a name."""

    return _NON_ALPHANUMERIC.sub("", column.casefold())


def _column_tokens(column: str) -> frozenset[str]:
    """Split snake case, punctuation, and camel case into lowercase tokens."""

    separated = _CAMEL_CASE_BOUNDARY.sub("_", column)
    return frozenset(re.findall(r"[a-z0-9]+", separated.casefold()))


def _forbidden_reason(column: str) -> str | None:
    """Return the leakage reason for a column, or ``None`` when not detected."""

    normalized = _normalize_column_name(column)
    tokens = _column_tokens(column)

    if (
        normalized in _LIST_PRICE_EXACT_ALIASES
        or any(marker in normalized for marker in _LIST_PRICE_MARKERS)
    ):
        return "list-price data and features derived from it are unavailable off market"

    if (
        normalized in {"dom", "cdom", "cumulativedom"}
        or tokens.intersection({"dom", "cdom"})
        or any(marker in normalized for marker in _DOM_MARKERS)
    ):
        return "days-on-market data and cumulative DOM variants are listing-only"

    if any(marker in normalized for marker in _CONTRACT_DATE_MARKERS):
        return "listing and purchase contract dates are transaction-only inputs"

    if any(marker in normalized for marker in _CLOSE_DATE_MARKERS):
        return (
            "CloseDate is metadata only; derive point-in-time AsOf* features "
            "before model input"
        )

    # The exact names are safe because their construction contract requires
    # eligible historical comps, subject exclusion, and an as-of cutoff.
    if column in _SAFE_COMP_PRICE_COLUMNS:
        return None

    if (
        any(marker in normalized for marker in _TARGET_PRICE_MARKERS)
        or "target" in tokens
        or normalized in {"target", "label", "response", "y", "sp"}
    ):
        return "ClosePrice is the target only; row-level target-derived inputs leak"

    if any(marker in normalized for marker in _PPSF_MARKERS):
        return (
            "generic price-per-square-foot inputs have unverified target provenance; "
            "use the allowlisted prior-comp aggregate"
        )

    if any(marker in normalized for marker in _CLOSE_TO_LIST_MARKERS):
        return "sale-to-list and close-to-list transformations leak listing data"

    return None


def validate_model_features(columns: Iterable[str]) -> None:
    """Validate a collection of model input column names.

    A valid input must be present in :data:`MODEL_FEATURE_COLUMNS` and must not
    match a known leakage alias or derived-feature pattern. A
    :class:`FeaturePolicyError` describes all violations in one message.

    Args:
        columns: Model input column names.

    Raises:
        TypeError: If ``columns`` is a string or contains a non-string value.
        FeaturePolicyError: If any input is forbidden or is not allowlisted.
    """

    if isinstance(columns, (str, bytes)):
        raise TypeError("columns must be an iterable of column-name strings, not a string")

    try:
        requested_columns = tuple(columns)
    except TypeError as exc:
        raise TypeError("columns must be an iterable of column-name strings") from exc

    non_string_columns = [column for column in requested_columns if not isinstance(column, str)]
    if non_string_columns:
        rendered = ", ".join(repr(column) for column in non_string_columns)
        raise TypeError(f"every model feature name must be a string; received: {rendered}")

    forbidden: list[tuple[str, str]] = []
    non_allowlisted: list[str] = []

    for column in requested_columns:
        reason = _forbidden_reason(column)
        if reason is not None:
            forbidden.append((column, reason))
        elif column not in _MODEL_FEATURE_COLUMN_SET:
            non_allowlisted.append(column)

    if not forbidden and not non_allowlisted:
        return

    message_lines = [
        "Model feature policy violation: every input must be explicitly allowlisted "
        "and available for an arbitrary off-market home at prediction time."
    ]

    if forbidden:
        message_lines.append("Forbidden or leakage-prone columns:")
        message_lines.extend(
            f"  - {column}: {reason}" for column, reason in forbidden
        )

    if non_allowlisted:
        message_lines.append("Non-allowlisted columns:")
        message_lines.extend(f"  - {column}" for column in non_allowlisted)

    message_lines.append(
        "ClosePrice may be used only as the target, and CloseDate may be used only "
        "as metadata or as the historical proxy used to derive AsOf* features."
    )
    raise FeaturePolicyError("\n".join(message_lines))


In [3]:
import math
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder

TARGET_COLUMN = "ClosePrice"


DATE_COLUMN = "CloseDate"


NUMERIC_FEATURE_COLUMNS = [
    "LivingArea",
    "AboveGradeFinishedArea",
    "BelowGradeFinishedArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "PropertyAge",
    "Latitude",
    "Longitude",
    "AssociationFee",
    "GarageSpaces",
    "ParkingTotal",
    "FireplacesTotal",
    "Stories",
    "MainLevelBedrooms",
    "AsOfYear",
    "AsOfMonthSin",
    "AsOfMonthCos",
    "CompMedianClosePrice",
    "CompMedianClosePricePerSqFt",
    "CompSaleCount",
    "CompDaysSinceLatestSale",
    "GeoCompMedianClosePrice",
    "GeoCompMedianClosePricePerSqFt",
    "GeoCompSaleCount",
    "GeoCompDaysSinceLatestSale",
    "ShiftGeoCompMedianClosePrice",
    "ShiftGeoCompMedianClosePricePerSqFt",
    "ShiftGeoCompSaleCount",
    "ShiftGeoCompDaysSinceLatestSale",
]


CATEGORICAL_FEATURE_COLUMNS = [
    "ViewYN",
    "WaterfrontYN",
    "BasementYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
    "CountyOrParish",
    "City",
    "PostalCode",
    "ElementarySchoolDistrict",
    "MiddleOrJuniorSchoolDistrict",
    "HighSchoolDistrict",
    "Levels",
]


RAW_NUMERIC_COLUMNS = [
    column
    for column in NUMERIC_FEATURE_COLUMNS
    if column
    not in {
        "PropertyAge",
        "AsOfYear",
        "AsOfMonthSin",
        "AsOfMonthCos",
        "CompMedianClosePrice",
        "CompMedianClosePricePerSqFt",
        "CompSaleCount",
        "CompDaysSinceLatestSale",
        "GeoCompMedianClosePrice",
        "GeoCompMedianClosePricePerSqFt",
        "GeoCompSaleCount",
        "GeoCompDaysSinceLatestSale",
        "ShiftGeoCompMedianClosePrice",
        "ShiftGeoCompMedianClosePricePerSqFt",
        "ShiftGeoCompSaleCount",
        "ShiftGeoCompDaysSinceLatestSale",
    }
]


SOURCE_COLUMNS = sorted(
    set(
        RAW_NUMERIC_COLUMNS
        + CATEGORICAL_FEATURE_COLUMNS
        + [
            TARGET_COLUMN,
            DATE_COLUMN,
            "ListingContractDate",
            "ListingKey",
            "PropertyType",
            "PropertySubType",
            "StateOrProvince",
            "LotSizeAcres",
            "latfilled",
            "lonfilled",
        ]
    )
)


MIN_CLOSE_PRICE = 50_000.0


MAX_CLOSE_PRICE = 30_000_000.0


MIN_LIVING_AREA = 200.0


MAX_LIVING_AREA = 20_000.0


CURRENT_MODEL_PARAMETERS = {
    "learning_rate": 0.05,
    "max_iter": 500,
    "max_leaf_nodes": 63,
    "min_samples_leaf": 20,
}


def _normalize_string_series(series: pd.Series) -> pd.Series:
    result = series.astype("string").str.strip()
    result = result.mask(result.eq(""))
    # scikit-learn 1.3 does not consistently accept pandas.NA in object arrays.
    return result.astype(object).where(result.notna(), np.nan)


def _record_filter(
    frame: pd.DataFrame,
    mask: pd.Series,
    audit: Dict[str, int],
    audit_key: str,
) -> pd.DataFrame:
    mask = mask.fillna(False)
    audit[audit_key] = int((~mask).sum())
    return frame.loc[mask].copy()


def discover_sales_files(data_dir: Path, pattern: str) -> List[Path]:
    """Return sorted sales files and fail clearly when the input is empty."""
    files = sorted(data_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No sales files matched {pattern!r} under {str(data_dir)!r}."
        )
    return files


def load_sales_files(
    files: Sequence[Path], max_rows: Optional[int] = None
) -> pd.DataFrame:
    """Load only columns needed for cleaning, feature creation, and the target."""
    frames: List[pd.DataFrame] = []
    remaining = max_rows
    string_dtypes = {
        "PostalCode": "string",
        "ListingKey": "string",
        "StateOrProvince": "string",
        "PropertyType": "string",
        "PropertySubType": "string",
    }

    for file_path in files:
        if remaining is not None and remaining <= 0:
            break
        frame = pd.read_csv(
            file_path,
            usecols=lambda column: column in SOURCE_COLUMNS,
            dtype=string_dtypes,
            low_memory=False,
            nrows=remaining,
        )
        frame["SourceFile"] = file_path.name
        frames.append(frame)
        if remaining is not None:
            remaining -= len(frame)

    if not frames:
        raise ValueError("The selected files contained no rows.")
    combined = pd.concat(frames, ignore_index=True, sort=False)
    for column in SOURCE_COLUMNS:
        if column not in combined:
            combined[column] = np.nan
    return combined


def clean_training_sales(
    raw_frame: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """Create the single-family California training population.

    ListingContractDate is used only for a chronology quality-control check. It is
    excluded at the model-feature boundary and never reaches the estimator.
    """
    frame = raw_frame.copy()
    audit: Dict[str, int] = {"rows_loaded": int(len(frame))}

    for column in [
        "StateOrProvince",
        "PropertyType",
        "PropertySubType",
        "PostalCode",
        "ListingKey",
    ] + CATEGORICAL_FEATURE_COLUMNS:
        if column in frame:
            frame[column] = _normalize_string_series(frame[column])

    for column in RAW_NUMERIC_COLUMNS + [
        TARGET_COLUMN,
        "LotSizeAcres",
        "latfilled",
        "lonfilled",
    ]:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors="coerce").replace(
                [np.inf, -np.inf], np.nan
            )

    frame[DATE_COLUMN] = pd.to_datetime(frame.get(DATE_COLUMN), errors="coerce")
    frame["ListingContractDate"] = pd.to_datetime(
        frame.get("ListingContractDate"), errors="coerce"
    )

    if "latfilled" in frame:
        frame["Latitude"] = frame["Latitude"].fillna(frame["latfilled"])
    if "lonfilled" in frame:
        frame["Longitude"] = frame["Longitude"].fillna(frame["lonfilled"])
    if "LotSizeAcres" in frame:
        acre_square_feet = frame["LotSizeAcres"] * 43_560.0
        frame["LotSizeSquareFeet"] = frame["LotSizeSquareFeet"].fillna(
            acre_square_feet
        )

    frame = _record_filter(
        frame,
        frame["StateOrProvince"].str.upper().eq("CA"),
        audit,
        "removed_non_california",
    )
    frame = _record_filter(
        frame,
        frame["PropertyType"].eq("Residential")
        & frame["PropertySubType"].eq("SingleFamilyResidence"),
        audit,
        "removed_non_single_family",
    )
    frame = _record_filter(
        frame,
        frame[DATE_COLUMN].notna() & frame[TARGET_COLUMN].notna(),
        audit,
        "removed_missing_date_or_target",
    )
    frame = _record_filter(
        frame,
        frame[TARGET_COLUMN].between(MIN_CLOSE_PRICE, MAX_CLOSE_PRICE),
        audit,
        "removed_implausible_price",
    )
    frame = _record_filter(
        frame,
        frame["LivingArea"].between(MIN_LIVING_AREA, MAX_LIVING_AREA),
        audit,
        "removed_implausible_living_area",
    )

    chronology_is_valid = frame["ListingContractDate"].isna() | (
        frame[DATE_COLUMN] >= frame["ListingContractDate"]
    )
    frame = _record_filter(
        frame,
        chronology_is_valid,
        audit,
        "removed_close_before_listing",
    )

    if "ListingKey" in frame:
        before_deduplication = len(frame)
        has_key = frame["ListingKey"].notna()
        keyed = frame.loc[has_key].drop_duplicates("ListingKey", keep="last")
        unkeyed = frame.loc[~has_key]
        frame = pd.concat([keyed, unkeyed], ignore_index=True, sort=False)
        audit["removed_duplicate_listing_key"] = int(
            before_deduplication - len(frame)
        )

    invalid_coordinates = ~(
        frame["Latitude"].between(32.0, 42.2)
        & frame["Longitude"].between(-124.7, -114.0)
    )
    audit["coordinates_set_missing"] = int(invalid_coordinates.sum())
    frame.loc[invalid_coordinates, ["Latitude", "Longitude"]] = np.nan

    for column in [
        "AboveGradeFinishedArea",
        "BelowGradeFinishedArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "AssociationFee",
        "GarageSpaces",
        "ParkingTotal",
        "FireplacesTotal",
        "Stories",
        "MainLevelBedrooms",
    ]:
        frame.loc[frame[column] < 0, column] = np.nan

    close_year = frame[DATE_COLUMN].dt.year
    invalid_year = ~frame["YearBuilt"].between(1700, close_year + 1)
    frame.loc[invalid_year, "YearBuilt"] = np.nan

    frame = frame.sort_values(DATE_COLUMN, kind="mergesort").reset_index(drop=True)
    audit["rows_after_cleaning"] = int(len(frame))
    return frame, audit


def add_as_of_features(
    frame: pd.DataFrame, as_of_dates: pd.Series
) -> pd.DataFrame:
    """Add features derived from the caller-known valuation timestamp."""
    result = frame.copy()
    parsed_dates = pd.to_datetime(as_of_dates, errors="coerce")
    if parsed_dates.isna().any():
        raise ValueError("Every row must have a valid as-of date.")

    month = parsed_dates.dt.month.astype(float)
    result["AsOfYear"] = parsed_dates.dt.year.astype(float)
    result["AsOfMonthSin"] = np.sin(2.0 * math.pi * month / 12.0)
    result["AsOfMonthCos"] = np.cos(2.0 * math.pi * month / 12.0)
    result["PropertyAge"] = result["AsOfYear"] - pd.to_numeric(
        result["YearBuilt"], errors="coerce"
    )
    result.loc[result["PropertyAge"] < 0, "PropertyAge"] = np.nan
    return result


def make_estimator(
    random_state: int = 42,
    feature_columns: Optional[Sequence[str]] = None,
    model_parameters: Optional[Mapping[str, Any]] = None,
) -> Pipeline:
    """Create the preprocessing and regression pipeline."""
    selected_columns = list(
        MODEL_FEATURE_COLUMNS if feature_columns is None else feature_columns
    )
    validate_model_features(selected_columns)
    selected_names = set(selected_columns)
    numeric_columns = [
        column for column in NUMERIC_FEATURE_COLUMNS if column in selected_names
    ]
    categorical_columns = [
        column for column in CATEGORICAL_FEATURE_COLUMNS if column in selected_names
    ]
    if set(numeric_columns + categorical_columns) != selected_names:
        raise ValueError("Every selected feature must have a preprocessing route.")

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median", add_indicator=True, keep_empty_features=True
                ),
            )
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "target_encoder",
                TargetEncoder(
                    target_type="continuous",
                    cv=5,
                    shuffle=True,
                    random_state=random_state,
                ),
            ),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    selected_model_parameters = dict(
        CURRENT_MODEL_PARAMETERS if model_parameters is None else model_parameters
    )
    regressor = HistGradientBoostingRegressor(
        loss="squared_error",
        l2_regularization=1.0,
        early_stopping=False,
        random_state=random_state,
        **selected_model_parameters,
    )
    return Pipeline(
        steps=[("preprocessor", preprocessor), ("regressor", regressor)]
    )


def fit_estimator(
    features: pd.DataFrame,
    target: pd.Series,
    random_state: int = 42,
    model_parameters: Optional[Mapping[str, Any]] = None,
) -> Pipeline:
    """Fit on log price so extreme luxury sales do not dominate the objective."""
    validate_model_features(features.columns)
    estimator = make_estimator(
        random_state=random_state,
        feature_columns=features.columns,
        model_parameters=model_parameters,
    )
    estimator.fit(features, np.log1p(target.to_numpy(dtype=float)))
    return estimator


def predict_estimator(estimator: Pipeline, features: pd.DataFrame) -> np.ndarray:
    """Return non-negative dollar predictions from a log-target estimator."""
    validate_model_features(features.columns)
    log_predictions = estimator.predict(features)
    return np.maximum(0.0, np.expm1(log_predictions))


def regression_metrics(actual: Sequence[float], predicted: Sequence[float]) -> Dict[str, float]:
    """Return interpretable price-model metrics."""
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    absolute_error = np.abs(actual_array - predicted_array)
    percentage_error = absolute_error / actual_array
    return {
        "mae_dollars": float(mean_absolute_error(actual_array, predicted_array)),
        "rmse_dollars": float(
            np.sqrt(mean_squared_error(actual_array, predicted_array))
        ),
        "median_absolute_error_dollars": float(np.median(absolute_error)),
        "median_absolute_percentage_error": float(np.median(percentage_error)),
        "r2": float(r2_score(actual_array, predicted_array)),
        "within_10_percent": float(np.mean(percentage_error <= 0.10)),
        "within_20_percent": float(np.mean(percentage_error <= 0.20)),
    }

## 1. Load and clean sales

The shared cleaner restricts records to California residential single-family sales, drops missing/invalid labels or dates and implausible prices/areas, checks close versus listing chronology, and deduplicates `ListingKey`. It fills coordinates from the supplied fallback fields and lot square feet from acres. Optional invalid values become missing. The listing date and listing key are used only for cleaning and are excluded from all model features.

The Week 2 exploration kept positive-price outliers for description, so its scoped count can differ from this modeling population.

In [4]:
sales_files = discover_sales_files(ROOT / 'csv', 'CRMLSSold*.csv')
raw = load_sales_files(sales_files)
cleaned, cleaning_audit = clean_training_sales(raw)
del raw

print(f'Files: {len(sales_files)}')
print(f'Cleaned rows: {len(cleaned):,}')
print(f'Close dates: {cleaned.CloseDate.min().date()} to {cleaned.CloseDate.max().date()}')
display(pd.Series(cleaning_audit, name='rows').to_frame())

Files: 28
Cleaned rows: 308,941
Close dates: 2024-01-01 to 2026-04-30


,rows
rows_loaded,615707
removed_non_california,79
removed_non_single_family,305915
removed_missing_date_or_target,2
removed_implausible_price,162
removed_implausible_living_area,302
removed_close_before_listing,44
removed_duplicate_listing_key,262
coordinates_set_missing,2770
rows_after_cleaning,308941


## 2. Choose safe fields and missing-value actions

The baseline uses only the project's approved raw property fields plus age and as-of calendar features. Entirely absent fields are dropped based on months **before** the test month. For retained numerical fields, the pipeline learns medians from each training partition and adds missingness indicators. For categories, it fills missing values with `__MISSING__` and cross-fits target encoding within each training partition; validation and test categories use mappings learned solely from that partition's training rows. This makes categorical model inputs numeric without fitting on test outcomes.

No scaling is applied: the baseline is a histogram gradient-boosted tree, for which standardization does not improve the split representation. If a distance-based or regularized linear model replaces it, fit a scaler on training rows only.

In [5]:
month = cleaned['CloseDate'].dt.to_period('M')
test_month = month.max()
development_mask = month < test_month

with_as_of = add_as_of_features(cleaned, cleaned['CloseDate'])
baseline_candidates = [c for c in MODEL_FEATURE_COLUMNS
                       if c in set(RAW_NUMERIC_COLUMNS + CATEGORICAL_FEATURE_COLUMNS)
                       or c in {'PropertyAge', 'AsOfYear', 'AsOfMonthSin', 'AsOfMonthCos'}]
entirely_missing = [c for c in baseline_candidates
                    if with_as_of.loc[development_mask, c].isna().all()]
feature_columns = [c for c in baseline_candidates if c not in entirely_missing]
validate_model_features(feature_columns)
features = with_as_of.loc[:, feature_columns].copy()
target = cleaned['ClosePrice'].astype(float)
del with_as_of

missing = pd.DataFrame({
    'missing_rows': features.loc[development_mask].isna().sum(),
    'missing_pct': (100 * features.loc[development_mask].isna().mean()).round(2),
}).sort_values('missing_pct', ascending=False)
print(f'Test month: {test_month}; test rows: {(month == test_month).sum():,}')
print(f'Approved candidate fields: {len(baseline_candidates)}; dropped as entirely missing: {entirely_missing}')
print(f'Retained fields: {len(feature_columns)}')
display(missing.head(15))

Test month: 2026-04; test rows: 12,470
Approved candidate fields: 33; dropped as entirely missing: ['AboveGradeFinishedArea', 'FireplacesTotal', 'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict']
Retained fields: 29


,missing_rows,missing_pct
WaterfrontYN,296342,99.96
BelowGradeFinishedArea,294664,99.39
BasementYN,289364,97.60
MainLevelBedrooms,119245,40.22
AssociationFee,89502,30.19
HighSchoolDistrict,76821,25.91
Stories,37907,12.79
AttachedGarageYN,34175,11.53
PoolPrivateYN,28643,9.66
Levels,28233,9.52


## 3. Tune the training window without touching the test month

For each candidate X, train on exactly the X calendar months immediately preceding each validation month. January, February, and March before the final test month are separate validation folds. Rank candidates by the **mean monthly dollar MAE**, breaking ties in favor of shorter histories. The same model settings and feature rules are used for every candidate. The selection is best among the tried values for this baseline, not a universal optimum. The final test month is evaluated only after X is chosen.

In [6]:
CANDIDATE_MONTHS = [3, 6, 9, 12, 18, 24]
validation_months = [test_month - offset for offset in (3, 2, 1)]
MODEL_PARAMETERS = {
    'learning_rate': 0.08,
    'max_iter': 120,
    'max_leaf_nodes': 31,
    'min_samples_leaf': 40,
}

assert (month == test_month).any()
assert all((month == period).any() for period in validation_months)
assert (month == validation_months[0] - max(CANDIDATE_MONTHS)).any(), 'Insufficient history.'
display(pd.DataFrame({
    'month': sorted(month.unique().astype(str)),
    'rows': [int((month == p).sum()) for p in sorted(month.unique())],
}).tail(8))

,month,rows
20,2025-09,11428
21,2025-10,11999
22,2025-11,9705
23,2025-12,10434
24,2026-01,7463
25,2026-02,8945
26,2026-03,11545
27,2026-04,12470


In [7]:
validation_results = []
for validation_month in validation_months:
    validation_mask = month == validation_month
    for x_months in CANDIDATE_MONTHS:
        training_mask = (month >= validation_month - x_months) & (month < validation_month)
        assert training_mask.any() and not (training_mask & validation_mask).any()
        estimator = fit_estimator(
            features.loc[training_mask, feature_columns],
            target.loc[training_mask],
            random_state=42,
            model_parameters=MODEL_PARAMETERS,
        )
        predicted = predict_estimator(estimator, features.loc[validation_mask, feature_columns])
        scores = regression_metrics(target.loc[validation_mask], predicted)
        validation_results.append({
            'validation_month': str(validation_month),
            'training_months': x_months,
            'training_rows': int(training_mask.sum()),
            'validation_rows': int(validation_mask.sum()),
            'mae_dollars': scores['mae_dollars'],
            'median_ape_pct': 100 * scores['median_absolute_percentage_error'],
        })
        print(f'{validation_month} | X={x_months:2d} | '
              f'train={training_mask.sum():6,d} | MAE=${scores["mae_dollars"]:,.0f}')
        del estimator

validation_results = pd.DataFrame(validation_results)
window_summary = (validation_results.groupby('training_months', as_index=False)
                  .agg(mean_monthly_mae_dollars=('mae_dollars', 'mean'),
                       mean_monthly_median_ape_pct=('median_ape_pct', 'mean'))
                  .sort_values(['mean_monthly_mae_dollars', 'training_months'])
                  .reset_index(drop=True))
selected_months = int(window_summary.iloc[0]['training_months'])
display(window_summary.round(2))
print(f'Selected X = {selected_months} months')

C:\Users\Yingjie Huang\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Yingjie Huang\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.9_3.9.3568.0_x64__qbz5n2kfra8p0\lib\subprocess.py", line 505, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Program Files\WindowsApps\PythonSoftwareFounda

2026-01 | X= 3 | train=32,138 | MAE=$204,385


2026-01 | X= 6 | train=67,082 | MAE=$194,149


2026-01 | X= 9 | train=102,321 | MAE=$197,709


2026-01 | X=12 | train=129,856 | MAE=$194,893


...

2026-03 | X=12 | train=129,317 | MAE=$198,720


2026-03 | X=18 | train=190,716 | MAE=$197,695


2026-03 | X=24 | train=267,070 | MAE=$199,156


,training_months,mean_monthly_mae_dollars,mean_monthly_median_ape_pct
0,12,195779.07,9.20
1,18,196235.89,9.14
2,24,196707.89,9.11
3,9,197216.77,9.17
4,6,197866.00,9.28
5,3,204890.36,9.41


Selected X = 12 months


## 4. Final chronological split and one test evaluation

Refit on the selected X months ending immediately before the latest month. The test set is every retained sale in that latest month. This score is a closed-sale holdout for the Week 3 baseline; the repository's production model and its comparable-sale evaluation remain separate.

In [8]:
train_mask = (month >= test_month - selected_months) & (month < test_month)
test_mask = month == test_month
assert train_mask.any() and test_mask.any()
assert month.loc[train_mask].min() == test_month - selected_months
assert month.loc[train_mask].max() == test_month - 1
assert not (train_mask & test_mask).any()

X_train = features.loc[train_mask, feature_columns]
X_test = features.loc[test_mask, feature_columns]
y_train = target.loc[train_mask]
y_test = target.loc[test_mask]

final_estimator = fit_estimator(
    X_train, y_train, random_state=42, model_parameters=MODEL_PARAMETERS
)
test_predictions = predict_estimator(final_estimator, X_test)
test_scores = regression_metrics(y_test, test_predictions)

print(f'Train: {month.loc[train_mask].min()} through {month.loc[train_mask].max()} '
      f'({len(X_train):,} rows)')
print(f'Test: {test_month} ({len(X_test):,} rows)')
display(pd.Series({
    'MAE ($)': round(test_scores['mae_dollars']),
    'RMSE ($)': round(test_scores['rmse_dollars']),
    'Median absolute percentage error (%)': round(
        100 * test_scores['median_absolute_percentage_error'], 2),
    'Within 10% (%)': round(100 * test_scores['within_10_percent'], 2),
}, name='held-out test').to_frame())

Train: 2025-04 through 2026-03 (130,274 rows)
Test: 2026-04 (12,470 rows)


,held-out test
MAE ($),203538.00
RMSE ($),561617.00
Median absolute percentage error (%),9.07
Within 10% (%),53.76


## 5. Export a numeric, model-ready cleaned CSV

`csv/cleaned_sales.csv` contains **only the selected training rows and final test rows**. `CloseDate`, `ClosePrice`, and `DataSplit` are metadata/label columns; all other columns are numeric model inputs. The preprocessing transformer is fitted only on the selected training partition. Its `fit_transform` applies cross-fitted category encoding to training rows, while `transform` applies the learned mapping to test rows. The target encoder receives the same log-price target that the baseline estimator uses. The CSV is intentionally ignored by the repository's data-file Git policy, but is written in the local workspace.

In [9]:
preprocessor = make_estimator(
    random_state=42,
    feature_columns=feature_columns,
    model_parameters=MODEL_PARAMETERS,
).named_steps['preprocessor']
encoded_train = preprocessor.fit_transform(X_train, np.log1p(y_train.to_numpy()))
encoded_test = preprocessor.transform(X_test)
encoded_names = list(preprocessor.get_feature_names_out())
assert len(set(encoded_names)) == len(encoded_names)

def export_partition(encoded, dates, prices, split):
    numeric_frame = pd.DataFrame(encoded, columns=encoded_names)
    assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in numeric_frame.dtypes)
    assert np.isfinite(numeric_frame.to_numpy(dtype=float)).all()
    metadata = pd.DataFrame({
        'CloseDate': pd.to_datetime(dates).dt.strftime('%Y-%m-%d').to_numpy(),
        'ClosePrice': prices.to_numpy(dtype=float),
        'DataSplit': split,
    })
    return pd.concat([metadata, numeric_frame], axis=1)

exported = pd.concat([
    export_partition(encoded_train, cleaned.loc[train_mask, 'CloseDate'], y_train, 'train'),
    export_partition(encoded_test, cleaned.loc[test_mask, 'CloseDate'], y_test, 'test'),
], ignore_index=True)
output_path = ROOT / 'csv' / 'cleaned_sales.csv'
exported.to_csv(output_path, index=False, float_format='%.12g')

assert exported['DataSplit'].value_counts().to_dict() == {
    'train': len(X_train), 'test': len(X_test)
}
print(f'Wrote {output_path} ({len(exported):,} rows, '
      f'{len(encoded_names)} numeric model columns)')
display(exported.groupby('DataSplit').agg(
    rows=('ClosePrice', 'size'), first_date=('CloseDate', 'min'), last_date=('CloseDate', 'max')
))

Wrote C:\Users\Yingjie Huang\Downloads\idx-exchange\csv\cleaned_sales.csv (142,744 rows, 41 numeric model columns)


,rows,first_date,last_date
DataSplit,,,
test,12470,2026-04-01,2026-04-30
train,130274,2025-04-01,2026-03-31


### Interpretation

Window selection uses only months before the held-out test. The median and category encoders are refitted for the final training window, and no target or date column enters the model input matrix. Numeric missingness indicators preserve potentially useful missing-value information. The exported CSV has no raw listing, status, or sale-process fields. Re-running the notebook with newer sold files automatically moves the held-out month and repeats window selection.